In [6]:
import os
from dotenv import load_dotenv
from langchain.agents import initialize_agent, AgentType, Tool
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chains import LLMMathChain
from langchain_openai import ChatOpenAI

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")
LLM_TEMPERATURE = float(os.getenv("LLM_TEMPERATURE", "0.2"))
MAX_AGENT_STEPS = int(os.getenv("MAX_AGENT_STEPS", "10"))

In [27]:
# ========== 模型和 Agent 初始化 ==========

# 初始化 LLM
llm = ChatOpenAI(
            model=LLM_MODEL,
            temperature=0.3,
            api_key=OPENAI_API_KEY,
            base_url=OPENAI_BASE_URL or None
        )
reasoning_template = """You are a reasoning agent tasked with solving the user's logic-based questions.\n
    Logically arrive at the solution, and be factual. In your answers, clearly detail the steps
    in bullet points and give the final answer.\n
    """
reasoning_prompt = ChatPromptTemplate.from_messages([
    ("system",
    reasoning_template),
    ("human", "Question: {question}")
])
reasoning_chain = reasoning_prompt | llm | StrOutputParser()

def reasoning_func(q: str) -> str:
    return reasoning_chain.invoke({"question": q})

reasoning_tool = Tool.from_function(
    name="Reasoning Tool",
    func=reasoning_func,
    description="Answer logic/reasoning questions. Input should be the raw question text."
)

math_chain = LLMMathChain.from_llm(llm=llm, verbose=True)
math_tool = Tool.from_function(
    name="Calculator",
    func=math_chain.run,
    description="用于数值计算和表达式化简"
)

wikipedia = WikipediaAPIWrapper()
wikipedia_tool = Tool(
    name="Wikipedia",
    func=wikipedia.run,
    description="A useful tool for searching the Internet to find information on world events, issues, dates, "
                "years, etc. Worth using for general topics. Use precise questions.",
)

# 初始化 Agent，带 verbose 日志
agent = initialize_agent(
    tools=[math_tool, wikipedia_tool, reasoning_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,          # 打印每一步
    handle_parsing_errors=True,
    max_iterations=5,      # 最多 5 步
    return_intermediate_steps=True
)

In [28]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a math teaching assistant. Based on the user's question, the reasoning steps "
     "(intermediate steps), and the final result, generate a well-structured standard solution.\n"
     "Requirements:\n"
     "1. Clearly explain the reasoning process step by step (formula derivation / tool usage).\n"
     "2. On the last line, explicitly state the final answer."),
    ("human", 
     "Question: {question}\n\nSteps: {steps}\n\nFinal Result: {answer}\n\nPlease generate the standard solution:")
])

summary_chain = summary_prompt | llm

class SummaryAgent:
    def __init__(self, chain):
        self.chain = chain

    def preprocess(self, result: dict) -> dict:
        """把 agent.invoke 的 result 转换成 {question, steps, answer}"""
        steps_text = "\n".join(
            f"Tool: {a.tool}, Input: {a.tool_input}, Output: {o}"
            for a, o in result.get("intermediate_steps", [])
        )
        return {
            "question": result.get("input", ""),
            "steps": steps_text,
            "answer": result.get("output", "")
        }

    def invoke(self, result: dict):
        processed = self.preprocess(result)
        return self.chain.invoke(processed)

# 初始化
summary_agent = SummaryAgent(summary_chain)

In [29]:
query = """
某工厂生产零件，每天生产的数量遵循等差数列：

第一天生产 120 个，

第二天生产 135 个，

第三天生产 150 个，
以此类推。

问：

第 20 天工厂能生产多少个零件？

前 20 天一共能生产多少个零件？
"""
result = agent.invoke(query)
summary = summary_agent.invoke(result)
print(summary.content)



> Entering new AgentExecutor chain...
我需要计算第20天工厂生产的零件数量，以及前20天的总生产量。根据题目，零件生产数量遵循等差数列，第一天生产120个，第二天135个，第三天150个。可以看出，公差为15。

首先，我可以使用等差数列的公式来计算第20天的生产数量：
\[ a_n = a_1 + (n - 1) \cdot d \]
其中：
- \( a_n \) 是第n天的生产数量
- \( a_1 \) 是第一天的生产数量（120个）
- \( d \) 是公差（15个）
- \( n \) 是天数（20天）

接下来，我还需要计算前20天的总生产量，可以使用等差数列求和公式：
\[ S_n = \frac{n}{2} \cdot (a_1 + a_n) \]
其中：
- \( S_n \) 是前n天的总生产量
- \( n \) 是天数（20天）

我将首先计算第20天的生产数量。

Action: Calculator  
Action Input: 120 + (20 - 1) * 15  

> Entering new LLMMathChain chain...
120 + (20 - 1) * 15```text
120 + (20 - 1) * 15
```
...numexpr.evaluate("120 + (20 - 1) * 15")...

Answer: 405
> Finished chain.

Observation: Answer: 405
Thought:我已经计算出第20天工厂生产的零件数量为405个。接下来，我需要计算前20天的总生产量。

首先，我已经知道第20天的生产数量是405个。现在我将使用等差数列求和公式来计算前20天的总生产量。

Action: Calculator  
Action Input: (20 / 2) * (120 + 405)  

> Entering new LLMMathChain chain...
(20 / 2) * (120 + 405)```text
(20 / 2) * (120 + 405)
```
...numexpr.evaluate("(20 / 2) * (120 + 405)")...

Answer: 5250.0
>

In [26]:
result

{'input': '\n某工厂生产零件，每天生产的数量遵循等差数列：\n\n第一天生产 120 个，\n\n第二天生产 135 个，\n\n第三天生产 150 个，\n以此类推。\n\n问：\n\n第 20 天工厂能生产多少个零件？\n\n前 20 天一共能生产多少个零件？\n',
 'output': '第 20 天工厂能生产 405 个零件，前 20 天一共能生产 5250 个零件。',
 'intermediate_steps': [(AgentAction(tool='Calculator', tool_input='135 - 120', log='我需要先确定这个等差数列的公差，然后计算第 20 天的生产数量以及前 20 天的总生产数量。等差数列的公差可以通过第二天和第一天的生产数量之差来计算。\n\nAction: Calculator  \nAction Input: 135 - 120  '),
   'Answer: 15'),
  (AgentAction(tool='Calculator', tool_input='120 + (20 - 1) * 15', log='我已经确定了公差为 15。接下来，我可以使用等差数列的公式来计算第 20 天的生产数量。等差数列的第 n 项可以用公式 \\( a_n = a_1 + (n - 1) \\cdot d \\) 来计算，其中 \\( a_1 \\) 是第一项，\\( d \\) 是公差，\\( n \\) 是项数。\n\nAction: Calculator  \nAction Input: 120 + (20 - 1) * 15  '),
   'Answer: 405'),
  (AgentAction(tool='Calculator', tool_input='(20 / 2) * (120 + 405)', log='我已经计算出第 20 天工厂能生产 405 个零件。接下来，我需要计算前 20 天的总生产数量。等差数列的前 n 项和可以用公式 \\( S_n = \\frac{n}{2} \\cdot (a_1 + a_n) \\) 来计算，其中 \\( S_n \\) 是前 n 项和，\\( a_1 \\) 是第一项，\\( a_n \\) 是第 n 项，\\( n \\) 是项